In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q transformers datasets torch scikit-learn openpyxl joblib

import pandas as pd
import numpy as np
import torch
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Load dataset
path = "/content/drive/MyDrive/colab_datasets/hate_speech.xlsx"
df = pd.read_excel(path)

# Clean
df = df.dropna()
df["Label"] = df["Label"].str.strip().str.lower()
df["Label"] = df["Label"].replace({"n":"no","on":"no"})
df["Label"] = df["Label"].map({"yes":1,"no":0})
df = df.dropna(subset=["Label"])

# Split (80-10-10)
X_train, X_temp, y_train, y_temp = train_test_split(
    df["Text"], df["Label"], test_size=0.2, stratify=df["Label"], random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Save path
SAVE_PATH = "/content/drive/MyDrive/colab_models"
os.makedirs(SAVE_PATH, exist_ok=True)

# Create folder safely
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

print("Save path ready:", SAVE_PATH)

Save path ready: /content/drive/MyDrive/colab_models


In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# Ensure folder exists (important)
os.makedirs(SAVE_PATH, exist_ok=True)

vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=10000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

joblib.dump(vectorizer, f"{SAVE_PATH}/tfidf.pkl")

print("TF-IDF saved successfully")

TF-IDF saved successfully


In [ ]:
# SVM

from sklearn.svm import LinearSVC

os.makedirs(SAVE_PATH, exist_ok=True)

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

def eval_model(model, X, y, name):
    pred = model.predict(X)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y, pred))
    print("F1:", f1_score(y, pred))
    print(classification_report(y, pred))

eval_model(svm_model, X_train_tfidf, y_train, "TRAIN")
eval_model(svm_model, X_val_tfidf, y_val, "VALIDATION")
eval_model(svm_model, X_test_tfidf, y_test, "TEST")

joblib.dump(svm_model, f"{SAVE_PATH}/svm.pkl")


TRAIN
Accuracy: 0.9934462042599672
F1: 0.9909021986353298
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2334
           1       1.00      0.98      0.99      1328

    accuracy                           0.99      3662
   macro avg       0.99      0.99      0.99      3662
weighted avg       0.99      0.99      0.99      3662


VALIDATION
Accuracy: 0.6855895196506551
F1: 0.49295774647887325
              precision    recall  f1-score   support

           0       0.72      0.84      0.77       292
           1       0.59      0.42      0.49       166

    accuracy                           0.69       458
   macro avg       0.66      0.63      0.63       458
weighted avg       0.67      0.69      0.67       458


TEST
Accuracy: 0.6462882096069869
F1: 0.4873417721518987
              precision    recall  f1-score   support

           0       0.71      0.75      0.73       292
           1       0.51      0.46      0.49       166



['/content/drive/MyDrive/colab_models/svm.pkl']

In [ ]:
# Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)

eval_model(rf_model, X_train_tfidf, y_train, "TRAIN")
eval_model(rf_model, X_val_tfidf, y_val, "VALIDATION")
eval_model(rf_model, X_test_tfidf, y_test, "TEST")

joblib.dump(rf_model, f"{SAVE_PATH}/rf.pkl")


TRAIN
Accuracy: 0.9997269251774986
F1: 0.9996233521657251
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2334
           1       1.00      1.00      1.00      1328

    accuracy                           1.00      3662
   macro avg       1.00      1.00      1.00      3662
weighted avg       1.00      1.00      1.00      3662


VALIDATION
Accuracy: 0.6877729257641921
F1: 0.33488372093023255
              precision    recall  f1-score   support

           0       0.68      0.96      0.80       292
           1       0.73      0.22      0.33       166

    accuracy                           0.69       458
   macro avg       0.71      0.59      0.57       458
weighted avg       0.70      0.69      0.63       458


TEST
Accuracy: 0.6921397379912664
F1: 0.3380281690140845
              precision    recall  f1-score   support

           0       0.68      0.96      0.80       292
           1       0.77      0.22      0.34       166



['/content/drive/MyDrive/colab_models/rf.pkl']

In [ ]:
# BERT

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": list(X_train), "label": list(y_train)}).map(tokenize, batched=True)
val_ds   = Dataset.from_dict({"text": list(X_val), "label": list(y_val)}).map(tokenize, batched=True)
test_ds  = Dataset.from_dict({"text": list(X_test), "label": list(y_test)}).map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_ds.set_format(type="torch", columns=["input_ids","attention_mask","label"])
test_ds.set_format(type="torch", columns=["input_ids","attention_mask","label"])

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./bert",
    num_train_epochs=10,
    learning_rate=2e-5,   # BEST for F1
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluation
print("\nTRAIN")
print(classification_report(y_train, np.argmax(trainer.predict(train_ds).predictions, axis=1)))

print("\nVALIDATION")
print(classification_report(y_val, np.argmax(trainer.predict(val_ds).predictions, axis=1)))

print("\nTEST")
print(classification_report(y_test, np.argmax(trainer.predict(test_ds).predictions, axis=1)))

# SAVE
trainer.save_model(f"{SAVE_PATH}/bert")
tokenizer.save_pretrained(f"{SAVE_PATH}/bert")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3662 [00:00<?, ? examples/s]

...

              precision    recall  f1-score   support

           0       0.71      0.97      0.82      2334
           1       0.84      0.32      0.46      1328

    accuracy                           0.73      3662
   macro avg       0.78      0.64      0.64      3662
weighted avg       0.76      0.73      0.69      3662


VALIDATION


              precision    recall  f1-score   support

           0       0.70      0.98      0.82       292
           1       0.86      0.27      0.41       166

    accuracy                           0.72       458
   macro avg       0.78      0.62      0.61       458
weighted avg       0.76      0.72      0.67       458


TEST


              precision    recall  f1-score   support

           0       0.71      0.95      0.81       292
           1       0.76      0.31      0.44       166

    accuracy                           0.72       458
   macro avg       0.74      0.63      0.63       458
weighted avg       0.73      0.72      0.68       458



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/colab_models/bert/tokenizer_config.json',
 '/content/drive/MyDrive/colab_models/bert/tokenizer.json')

In [ ]:
# BERT + Class Weights

from torch import nn

class_weights = torch.tensor([
    len(y_train)/(2*(y_train==0).sum()),
    len(y_train)/(2*(y_train==1).sum())
]).float()

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

model_w = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

trainer_w = WeightedTrainer(
    model=model_w,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer_w.train()

# Evaluation
print("\nTEST (Weighted)")
print(classification_report(y_test, np.argmax(trainer_w.predict(test_ds).predictions, axis=1)))

trainer_w.save_model(f"{SAVE_PATH}/bert_weighted")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.602124,0.672489,0.539877
2,No log,0.612287,0.696507,0.512281
3,0.607948,0.680617,0.727074,0.555160
4,0.607948,0.822923,0.705240,0.526316
5,0.408461,0.986343,0.620087,0.558376
6,0.408461,1.371804,0.687773,0.508591
7,0.219431,1.596069,0.707424,0.528169
8,0.219431,1.812994,0.720524,0.518797
9,0.108252,1.915398,0.709607,0.519856
10,0.108252,1.876763,0.716157,0.563758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


TEST (Weighted)


              precision    recall  f1-score   support

           0       0.75      0.77      0.76       292
           1       0.58      0.55      0.57       166

    accuracy                           0.69       458
   macro avg       0.67      0.66      0.66       458
weighted avg       0.69      0.69      0.69       458



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# XLM-RoBERTa

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer_xlm = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tok_xlm(batch):
    return tokenizer_xlm(batch["text"], padding="max_length", truncation=True, max_length=128)

train_xlm = Dataset.from_dict({"text": list(X_train), "label": list(y_train)}).map(tok_xlm, batched=True)
val_xlm   = Dataset.from_dict({"text": list(X_val), "label": list(y_val)}).map(tok_xlm, batched=True)
test_xlm  = Dataset.from_dict({"text": list(X_test), "label": list(y_test)}).map(tok_xlm, batched=True)

train_xlm.set_format(type="torch", columns=["input_ids","attention_mask","label"])
val_xlm.set_format(type="torch", columns=["input_ids","attention_mask","label"])
test_xlm.set_format(type="torch", columns=["input_ids","attention_mask","label"])

model_xlm = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=2)

trainer_xlm = Trainer(
    model=model_xlm,
    args=training_args,
    train_dataset=train_xlm,
    eval_dataset=val_xlm,
    compute_metrics=compute_metrics
)

trainer_xlm.train()

print("\nTEST (XLM)")
print(classification_report(y_test, np.argmax(trainer_xlm.predict(test_xlm).predictions, axis=1)))

trainer_xlm.save_model(f"{SAVE_PATH}/xlm_roberta")
tokenizer_xlm.save_pretrained(f"{SAVE_PATH}/xlm_roberta")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3662 [00:00<?, ? examples/s]

...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


TEST (XLM)


              precision    recall  f1-score   support

           0       0.71      0.91      0.80       292
           1       0.69      0.36      0.47       166

    accuracy                           0.71       458
   macro avg       0.70      0.63      0.64       458
weighted avg       0.71      0.71      0.68       458



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/colab_models/xlm_roberta/tokenizer_config.json',
 '/content/drive/MyDrive/colab_models/xlm_roberta/tokenizer.json')

In [ ]:
# T5

from transformers import T5Tokenizer, T5ForConditionalGeneration

t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Convert labels to text
train_text = ["classify: "+t for t in X_train]
train_label = ["yes" if y==1 else "no" for y in y_train]

enc = t5_tokenizer(train_text, padding=True, truncation=True, return_tensors="pt")
labels = t5_tokenizer(train_label, padding=True, truncation=True, return_tensors="pt").input_ids

outputs = t5_model(**enc, labels=labels)
loss = outputs.loss

print("T5 initial loss:", loss.item())

# SAVE
t5_model.save_pretrained(f"{SAVE_PATH}/t5")
t5_tokenizer.save_pretrained(f"{SAVE_PATH}/t5")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
df_results.to_csv(f"{SAVE_PATH}/model_comparison.csv", index=False)